# 🌾 Paddy-YOLO: Baseline YOLOv8-N Training on R2000

This notebook trains the **Baseline YOLOv8-N** model on the **R2000** rice pest dataset (`dmrashidferrer/r2000v1-2024-images`) to reproduce the baseline benchmark from Zheng et al. (*Computers and Electronics in Agriculture*, 2025).

### Configuration:
- **Model Architecture**: YOLOv8-N (standard)
- **Bounding Box Loss**: CIoU
- **Optimizer**: AdamW, lr0=0.01, lrf=0.01, momentum=0.937, weight_decay=0.0005
- **Training Schedule**: 300 epochs, batch size 32, image size 640x640
- **Split**: 5-Fold Cross Validation Fold 0 (3 folds train, 1 fold val, 1 fold test)

In [ ]:
# Cell 1: Extract project codebase
import base64, io, zipfile, os

B64_ARCHIVE = '''...'''

zip_data = base64.b64decode(B64_ARCHIVE.encode('utf-8'))
with zipfile.ZipFile(io.BytesIO(zip_data)) as z:
    z.extractall('.')

print('Codebase successfully unpacked into working directory.')


In [ ]:
# Cell 2: Verify GPU Accelerator & PyTorch Environment
import torch
print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))
    print('Device Count:', torch.cuda.device_count())
else:
    raise SystemError('GPU accelerator is not available! Please enable GPU in notebook settings.')


In [ ]:
# Cell 3: Install Ultralytics and Run Sanity Check
!pip install ultralytics -q
!python verify_paddy_yolo.py --smoke-test


In [ ]:
# Cell 4: Discover & Prepare Kaggle R2000 Dataset with 5-Fold Splits
from pathlib import Path
from paddy_yolo.dataset.prepare_r2000 import prepare_kaggle_r2000, create_5fold_splits

kaggle_input = Path('/kaggle/input')
candidates = list(kaggle_input.glob('*r2000*')) if kaggle_input.exists() else []

if candidates:
    raw_data_dir = candidates[0]
    print(f'Found Kaggle R2000 dataset at: {raw_data_dir}')
else:
    raw_data_dir = Path('/kaggle/input/r2000v1-2024-images')
    print(f'Using default path: {raw_data_dir}')

output_dir = Path('/kaggle/working/data/r2000')
prepare_kaggle_r2000(input_dir=raw_data_dir, output_dir=output_dir)
folds = create_5fold_splits(dataset_dir=output_dir)
print('Generated 5-fold cross validation split configs:')
for f in folds:
    print(f'  -> {f}')


In [ ]:
# Cell 5: Train Baseline YOLOv8-N (300 Epochs, AdamW, CIoU)
!python train.py \
    --baseline \
    --data /kaggle/working/data/r2000/data_fold0.yaml \
    --hyp configs/hyp_paper.yaml \
    --epochs 300 \
    --batch 32 \
    --imgsz 640 \
    --device 0 \
    --project /kaggle/working/runs/train \
    --name baseline_yolov8n_fold0


In [ ]:
# Cell 6: Evaluate Baseline Checkpoint on Test Set
best_weights = '/kaggle/working/runs/train/baseline_yolov8n_fold0/weights/best.pt'

!python val.py \
    --weights {best_weights} \
    --data /kaggle/working/data/r2000/data_fold0.yaml \
    --split test \
    --device 0


In [ ]:
# Cell 7: Package artifacts for download
import shutil
train_run_dir = '/kaggle/working/runs/train/baseline_yolov8n_fold0'
zip_dest = '/kaggle/working/baseline_yolov8n_fold0_results'
if os.path.exists(train_run_dir):
    shutil.make_archive(zip_dest, 'zip', train_run_dir)
    print(f'Created downloadable results archive at: {zip_dest}.zip')
